# Challenger Model: LightGBM
### Flagship PD Model Project · Stage 3 (Challenger)

> **Objective.** Fit a gradient-boosted tree model (LightGBM) on the same 12 features, same rows and same train/test split as the baseline scorecard, and use it as a **benchmark**: how much signal, if any, is the explainable scorecard leaving on the table? The verdict decides whether the scorecard stays champion.

| Field | Detail |
|---|---|
| **Project** | Flagship PD Model - Project 3: Challenger Model |
| **Stage** | Stage 3 - Challenger benchmark |
| **Dataset** | Home Credit Default Risk - raw engineered features `train_raw` / `test_raw` (same rows as `train_woe` / `test_woe`) |
| **Target** | `TARGET` (1 = payment difficulty, 0 = repaid) |
| **Champion** | `logreg_baseline_v1` (test Gini 0.4926, KS 0.3620) |
| **Author** | Chuch |
| **AI assistance** | Built with AI assistance (Claude): notebook scaffold, cells marked "Written by AI", and editing of the write-ups. Predictions and final decisions are the author's. |
| **Date started / last updated** | *<YYYY-MM-DD>* / *<YYYY-MM-DD>* |
| **Notebook version** | *<v0.1 - draft>* |
| **Status** | In Progress |

**Reading guide - the E / O / D convention.** Each analysis block is written as three linked notes:
- **Explanation** - the hypothesis / business rationale *before* looking at the output.
- **Observation** - what the output actually showed.
- **Decision** - the action taken and its justification.

**Fair-comparison rule.** Only the model changes. Same 12 features, same rows, same test set, same metrics as notebook 02. The test set is never used for fitting, early stopping or tuning.

**Reproducibility.** *(placeholder - Python version, lightgbm version, `RANDOM_STATE`, input snapshot date)*

## Contents

- **Project Setup** - environment & imports, display settings, style guide, data loading
- **1. Scope & Inputs** - challenger question and success criteria · inherited inputs check
- **2. Data Preparation for Trees** - categorical handling · validation split for early stopping
- **3. LightGBM Model** - model setup · fit with early stopping · feature importance and direction check
- **4. Results vs Champion** - discrimination · train vs test gap · rank ordering · calibration
- **5. Verdict** - champion vs challenger
- **6. Findings Summary**
- **7. Assumptions, Limitations & Data Lineage**
- **8. Next Steps**
- **9. Save Model to Registry**

# Project Setup

## Environment and Imports

Same setup as notebook 02, plus `lightgbm`. If it is not installed in your venv: `pip install lightgbm`. Written by AI (copied from notebook 02).

In [ ]:
# Standard library
import os
import sys
import json
import pickle
import warnings
import joblib
import shutil

# Data
import numpy as np
import pandas as pd
import json

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

# Modelling
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import add_constant 
import sklearn


# Jupyter
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display


# Repo paths
def find_repo_root(start, repo_name='CreditRiskLearning'):
    path = os.path.abspath(start)
    while True:
        if os.path.basename(path) == repo_name:
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root: {repo_name}")
        path = parent


repo_root = find_repo_root(os.getcwd())
sys.path.append(os.path.join(repo_root, 'Core Resources'))
sys.path.append(os.path.join(repo_root, 'Core Resources', 'Scripts'))

# Style utils
from python_style_util import (
    palette, fonts, sizes,
    p, f, s,
    C, Shade, Size,
    set_style,
    gradient, traffic_light, diverging, diverging_cmap, risk_cmap, RISK_LOW, RISK_HIGH,
    multi_series, binary_palette,
    chart_title, annotate, style_df,
    show_palette, show_fonts, show_tokens, show_colormaps, show_style_guide,
)
import evaluation_utils as eu

# Project toolkit (credit_utils)
%load_ext autoreload
%autoreload 2

from credit_utils import eda, quality, woe, stats, style

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Display Settings

In [ ]:
# Pandas
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places on floats
pd.set_option('display.max_columns', None)              # show all columns
pd.set_option('display.max_rows', 100)                  # show up to 100 rows
pd.set_option('display.width', None)                    # don't wrap wide dataframes
pd.set_option('display.colheader_justify', 'left')      # left-align column headers
pd.set_option('display.precision', 4)                   # decimal precision for describe()
pd.set_option('display.large_repr', 'truncate')         # truncate instead of summary for large dfs

# NumPy
np.set_printoptions(
    precision=4,        # decimal places
    suppress=True,      # no scientific notation
    linewidth=120,      # wrap width
    threshold=1000,     # show up to 1000 elements before summarising with ...
    edgeitems=5,        # show 5 items at each end when it does summarise
)

# Matplotlib
%matplotlib inline
plt.rcParams['figure.dpi'] = 120             # sharper figures
plt.rcParams['savefig.dpi'] = 150            # higher res when saving
plt.rcParams['savefig.bbox'] = 'tight'       # no clipped labels when saving
plt.rcParams['savefig.facecolor'] = 'white'  # white background on saved figures

# Jupyter
InteractiveShell.ast_node_interactivity = 'all'  # print every expression, not just the last

# Warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Style
set_style()

## Style Guide

Quick visual reference for the project's colour tokens and fonts.

In [ ]:
# Colour tokens and fonts
show_tokens()
show_fonts()

## Data Loading

Loads the **raw** engineered features (trees split on raw values, so WOE is not needed), the champion's WOE frames and registry model (to score the champion on exactly the same rows), and the feature list. Written by AI.

In [ ]:
# Inputs from the driver scan / notebook 02
INPUT_DIR = os.path.join(repo_root, 'Projects', 'Flagship PD Model Project',
                         '02_PD_Model_Development', 'inputs')
REGISTRY_DIR = os.path.join(repo_root, 'Projects', 'Flagship PD Model Project',
                            '05_Applicant_PD_Dashboard', 'registry')

# Raw engineered features (89 columns) - the challenger's input
train_raw = pd.read_parquet(os.path.join(INPUT_DIR, 'train_raw.parquet'))
test_raw = pd.read_parquet(os.path.join(INPUT_DIR, 'test_raw.parquet'))

# WOE frames - only used to score the champion on the same rows
train_woe = pd.read_parquet(os.path.join(INPUT_DIR, 'train_woe.parquet'))
test_woe = pd.read_parquet(os.path.join(INPUT_DIR, 'test_woe.parquet'))

# Champion model and its feature list (same 12 features for the challenger)
CHAMP_DIR = os.path.join(REGISTRY_DIR, 'logreg_baseline_v1')
champion = joblib.load(os.path.join(CHAMP_DIR, 'model.joblib'))
with open(os.path.join(CHAMP_DIR, 'metadata.json')) as fh:
    champ_meta = json.load(fh)

model_features = champ_meta['features']
CATEGORICAL = ['OCCUPATION_TYPE', 'ORG_BAND', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE']

y_train, y_test = train_raw['TARGET'], test_raw['TARGET']

print(f"Train: {len(train_raw):,} rows | Test: {len(test_raw):,} rows | Features: {len(model_features)}")
print(f"Categorical: {CATEGORICAL}")

# Project Body

## 1. Scope & Inputs

### 1.1 Challenger Question & Success Criteria

- **Question:** does a flexible model that finds its own splits and interactions beat the WOE scorecard by enough to justify losing explainability?
- **Champion to beat:** test Gini **0.4926**, train Gini 0.4898, KS 0.3620 (`logreg_baseline_v1`).
- **Model scope:** one LightGBM model, sensible defaults plus early stopping. No large hyperparameter search: the point is a benchmark, not squeezing out every point.
- **Success criteria:** set each bar **before fitting** (fill in 1.1a to 1.1d). The champion stays unless the challenger clears the decision rule in 1.1a.

#### 1.1a Decision rule on test Gini uplift: *___*

*(Your decision. Template:)* Uplift = challenger test Gini minus 0.4926.
- Under *___* : scorecard **retained**; the gap is the price of explainability and is documented.
- *___* to *___* : scorecard retained, but investigate what the trees found (interactions, binning) for the next scorecard version.
- Over *___* : the scorecard is missing real signal; revisit features / binning before anything else.

*Reasoning:* *___* (consider: test Gini sampling noise is roughly ±0.01; what size of gap would a validator call material?)

#### 1.1b Maximum train/test Gini gap: *___*

*(Your decision.)* Trees can memorise far more than a 12-weight logistic model, so the champion's 0.02 limit may be too tight or just right. *Reasoning:* *___*

#### 1.1c Rank ordering: *___*

*(Your decision.)* Same test as 02 (strictly monotonic across test PD deciles, zero reversals) or something else? *Reasoning:* *___*

#### 1.1d Calibration: mean predicted PD within *___* pp of the actual test default rate

*(Your decision.)* No class weighting is used, so the average PD should land near the 8.07% base rate, as the champion's does. *Reasoning:* *___*

### 1.2 Inherited Inputs Check

**Explanation.** *(before running: same row counts as notebook 02 (train 246,005, test 61,502), raw and WOE frames in the same row order, same default rate)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: check row counts, that train_raw / train_woe (and test) share the same index and TARGET,
# and the default rate in each split. Hint: (a.index == b.index).all()

## 2. Data Preparation for Trees

### 2.1 Categorical Handling

LightGBM reads pandas `category` columns natively: no WOE, no one-hot encoding. The category **levels must be fixed from train** and reused for test and in the app, otherwise the same label can get a different internal code.

**Explanation.** *(before running: how many levels each of the 4 categorical features has, and whether any test level is unseen in train)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: build X_train / X_test with model_features.
# For each column in CATEGORICAL: fix the levels from train, e.g.
#   levels = sorted(X_train[col].dropna().unique())
#   X_train[col] = pd.Categorical(X_train[col], categories=levels)
#   X_test[col]  = pd.Categorical(X_test[col],  categories=levels)
# Keep the levels in a dict (cat_levels) - the registry adapter needs them later.

### 2.2 Validation Split for Early Stopping

Early stopping keeps adding trees until a **validation** set stops improving. That set must come out of **train**, never test, or the test Gini is no longer an honest out-of-sample number.

**Explanation.** *(before running: validation size and stratification choice, and the default rate in fit vs validation)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: split X_train / y_train into X_fit / y_fit and X_val / y_val
# Hint: train_test_split(..., test_size=___, stratify=y_train, random_state=RANDOM_STATE)

## 3. LightGBM Model

### 3.1 Model Setup

Each setting gets one line of reasoning, like 02's class weighting / regularisation / solver notes. Decide and justify:

- **learning_rate:** *___* (smaller = more trees, steadier)
- **num_leaves:** *___* (tree complexity; default 31)
- **min_child_samples:** *___* (minimum rows per leaf; guards against tiny, noisy leaves)
- **n_estimators + early stopping rounds:** *___*
- **Class weighting:** *___* (02 used none so PDs stay calibrated; same reasoning applies?)
- **Monotone constraints (optional):** *___* (force PD to fall as EXT scores rise? Explainability vs fit)

In [ ]:
# TODO: define the model
# model = lgb.LGBMClassifier(
#     n_estimators=___, learning_rate=___, num_leaves=___, min_child_samples=___,
#     random_state=RANDOM_STATE, verbose=-1,
# )

### 3.2 Fit with Early Stopping

**Explanation.** *(before running: roughly how many trees before validation AUC stops improving, and why)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: fit on the fit split, early-stop on the validation split
# model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], eval_metric='auc',
#           callbacks=[lgb.early_stopping(___), lgb.log_evaluation(100)])
# model.best_iteration_

### 3.3 Feature Importance & Direction Check

The scorecard's version of this was the coefficient sign check (3.3 in 02). Trees have no coefficients, so check (a) which features carry the most **gain**, and (b) that each feature moves PD the way credit logic says (e.g. higher EXT score, lower PD).

**Explanation.** *(before running: expected top features (compare with the scorecard's strongest coefficients / IV ranking) and expected directions)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: (a) importance by gain: model.booster_.feature_importance(importance_type='gain')
# (b) direction: for the numeric features, average predicted PD by decile of the feature,
#     or SHAP if you want to go further (pip install shap)

## 4. Results vs Champion

### 4.1 Discrimination (AUC / Gini, KS) - side by side

**Explanation.** *(before running: challenger test Gini relative to 0.4926 and your 1.1a bands)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# Champion PDs on the same test rows (Written by AI)
champ_pd_test = champion.predict_proba(test_woe[model_features])[:, 1]
champ_pd_train = champion.predict_proba(train_woe[model_features])[:, 1]

# TODO: challenger PDs (pd_test / pd_train) with model.predict_proba(...)[:, 1]
# then AUC, Gini = 2*AUC - 1, KS for both models in one small table

### 4.2 Train vs Test Gap

**Explanation.** *(before running: gap against your 1.1b limit)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: train Gini minus test Gini, both models

### 4.3 Rank Ordering by Score Band - side by side

**Explanation.** *(before running: decile 1 and decile 10 default rates for each model; any reversals)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: deciles of predicted PD (pd.qcut, q=10) for each model; n, actual rate, mean PD per decile

### 4.4 Calibration

**Explanation.** *(before running: mean predicted PD vs actual default rate for each model, against your 1.1d tolerance)*

**Observation.** *(after running)*

**Decision.** *(action and why)*

In [ ]:
# TODO: mean PD vs actual rate, overall and per decile (the decile table from 4.3 can carry both)

## 5. Verdict

**Criteria scorecard** *(fill in after section 4)*

| Criterion | Bar (from 1.1) | Champion | Challenger | Met? |
|---|---|---|---|---|
| 1.1a Test Gini uplift | | 0.4926 | | |
| 1.1b Train/test gap | | | | |
| 1.1c Rank ordering | | | | |
| 1.1d Calibration | | | | |

**Decision.** *___ stays champion because ___. The challenger is saved to the registry as a benchmark and shown alongside it on the dashboard.*

## 6. Findings Summary

*(placeholder)*

## 7. Assumptions, Limitations & Data Lineage

*(placeholder - same target proxy limitation as 02; same population incl. out-of-policy income types; no hyperparameter search; explainability limits of trees)*

## 8. Next Steps

*(placeholder - e.g. monotonic binning for the scorecard spikes, calibration by band, what the trees' top interactions suggest for a v2 scorecard)*

## 9. Save Model to Registry

Saves the challenger into its own registry folder so the dashboard picks it up. The folder needs `model.joblib`, the category levels from 2.1, and a `metadata.json` with `"role": "challenger"` and `"adapter": "lgbm_adapter"`. The adapter (`pd_app/lgbm_adapter.py`, same `score(row, model_dir)` contract as `scorecard.py`) is the next build step after this notebook.

In [ ]:
# TODO: model_dir = os.path.join(REGISTRY_DIR, 'lgbm_challenger_v1'); os.makedirs(model_dir, exist_ok=True)
# joblib.dump(model, ...); json.dump(cat_levels, ...)  -> category_levels.json
# metadata: name, version, role='challenger', adapter='lgbm_adapter', features, gini_test, gini_train,
#           ks_test, sklearn_version, lightgbm_version (use float() for numpy numbers)